# Week 4 exercise — Education tutor (LangGraph + OpenRouter)

LangGraph version of the same **education research** idea as the Week 2 Agents SDK notebook:






## Imports and OpenRouter `ChatOpenAI`

In [ ]:
import json
import os
import uuid
from typing import Annotated, Any, TypedDict

import httpx
import wikipedia
import gradio as gr
from dotenv import load_dotenv

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from decouple import config


OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"

OPENROUTER_API_KEY = config("OPENROUTER_API_KEY")
NEWS_API_KEY = config("NEWS_API_KEY")

llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    base_url=OPENROUTER_BASE,
    api_key=OPENROUTER_API_KEY,
    temperature=0,
)
print("Model:", OPENROUTER_MODEL)

## Tools 

In [ ]:
wikipedia.set_lang("en")


@tool
def wikipedia_summary(topic: str, sentences: int = 4) -> str:
    """Encyclopedic overview for stable facts, history, science, biographies. Not for breaking news."""
    topic = (topic or "").strip()
    if not topic:
        return json.dumps({"error": "empty topic"})
    try:
        page = wikipedia.page(topic, auto_suggest=True)
        summary = wikipedia.summary(
            topic, sentences=min(max(int(sentences), 1), 8), auto_suggest=True
        )
        return json.dumps(
            {"title": page.title, "url": page.url, "summary": summary}
        )
    except wikipedia.DisambiguationError as e:
        return json.dumps({"error": "disambiguation", "options": e.options[:8]})
    except Exception as e:
        return json.dumps({"error": type(e).__name__, "message": str(e)})


@tool
def news_search(query: str, page_size: int = 5) -> str:
    """Recent news for current events, today, this week, or breaking stories."""
    api_key = os.getenv("NEWS_API_KEY")
    if not api_key:
        return json.dumps(
            {
                "error": "NEWS_API_KEY not set",
                "hint": "Add NEWS_API_KEY from newsapi.org to .env.",
            }
        )
    query = (query or "").strip()
    if not query:
        return json.dumps({"error": "empty query"})
    page_size = max(1, min(int(page_size), 10))
    try:
        r = httpx.get(
            "https://newsapi.org/v2/everything",
            params={
                "q": query,
                "language": "en",
                "sortBy": "publishedAt",
                "pageSize": page_size,
                "apiKey": api_key,
            },
            timeout=30.0,
        )
        data = r.json()
        if data.get("status") != "ok":
            return json.dumps({"error": data.get("message", "newsapi error")})
        articles = []
        for a in data.get("articles", []):
            articles.append(
                {
                    "title": a.get("title"),
                    "url": a.get("url"),
                    "source": (a.get("source") or {}).get("name"),
                    "publishedAt": a.get("publishedAt"),
                    "description": a.get("description"),
                }
            )
        return json.dumps({"query": query, "articles": articles})
    except Exception as e:
        return json.dumps({"error": type(e).__name__, "message": str(e)})


@tool
def duckduckgo_instant_answer(query: str) -> str:
    """Quick instant-answer snippet when Wikipedia is weak. No API key."""
    query = (query or "").strip()
    if not query:
        return json.dumps({"error": "empty query"})
    try:
        r = httpx.get(
            "https://api.duckduckgo.com/",
            params={"q": query, "format": "json", "no_html": 1},
            timeout=20.0,
        )
        d = r.json()
        out = {
            "query": query,
            "abstract": d.get("Abstract") or "",
            "abstract_url": d.get("AbstractURL") or "",
            "heading": d.get("Heading") or "",
        }
        snippets = []
        for t in (d.get("RelatedTopics") or [])[:5]:
            if isinstance(t, dict) and "Text" in t:
                snippets.append({"text": t.get("Text"), "url": t.get("FirstURL")})
        out["related"] = snippets
        if not out["abstract"] and not snippets:
            out["note"] = "No instant answer; try wikipedia_summary."
        return json.dumps(out)
    except Exception as e:
        return json.dumps({"error": type(e).__name__, "message": str(e)})


tools = [wikipedia_summary, news_search, duckduckgo_instant_answer]
llm_with_tools = llm.bind_tools(tools)

## State graph — agent, `ToolNode`, `tools_condition`

In [ ]:
TUTOR_SYSTEM = SystemMessage(
    content=(
        "You are a helpful educational assistant.\n"
        "- Use news_search for timely/current events (today, this week, breaking news).\n"
        "- Use wikipedia_summary for stable concepts, history, science, biographies.\n"
        "- Use duckduckgo_instant_answer for quick facts or when other tools are weak.\n"
        "Ground answers in tool output; cite titles and URLs when present. "
        "If a tool returns JSON with an error field, explain it honestly."
    )
)


class AgentState(TypedDict):
    messages: Annotated[list[Any], add_messages]


def agent_node(state: AgentState) -> dict:
    msgs = state["messages"]
    if not msgs or not isinstance(msgs[0], SystemMessage):
        msgs = [TUTOR_SYSTEM] + list(msgs)
    else:
        msgs = list(msgs)
    response = llm_with_tools.invoke(msgs)
    return {"messages": [response]}


graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "tools", END: END},
)
graph_builder.add_edge("tools", "agent")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

## Smoke test (`invoke`)

In [ ]:
def last_assistant_text(messages: list) -> str:
    for m in reversed(messages):
        if isinstance(m, AIMessage) and (m.content or "").strip():
            return m.content
    return "(No assistant text)"


smoke_out = graph.invoke(
    {"messages": [HumanMessage(content="What is photosynthesis? Answer in 2–3 sentences.")]},
    config={"configurable": {"thread_id": "smoke-week4-edu"}},
)
print(last_assistant_text(smoke_out["messages"]))

## Gradio — one new `HumanMessage` per turn; thread id keeps context

### Sample prompts

1. **Wikipedia:** "Explain mitochondria for a high-school student."
2. **News** (needs `NEWS_API_KEY`): "Recent headlines about space exploration."
3. **Quick fact:** "What is the capital of Canada?"

In [ ]:
CHAT_THREAD_ID = str(uuid.uuid4())


def chat(message: str, history: list) -> str:
    result = graph.invoke(
        {"messages": [HumanMessage(content=message)]},
        config={"configurable": {"thread_id": CHAT_THREAD_ID}},
    )
    return last_assistant_text(result["messages"])


gr.ChatInterface(
    chat,
    type="messages",
    title="Education tutor (LangGraph + OpenRouter)",
).launch()